Ноут для теста гипотез

In [ ]:

import json
import os
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_PATH = Path("PERTA.parquet")
OUT_DIR = Path("hypothesis_outputs")
OUT_DIR.mkdir(exist_ok=True)
TRAIN_DAYS = 30
TEST_DAYS = 7
STEP_DAYS = 7
ANNUALIZATION_FACTOR = np.sqrt(24 * 365)
SIMPLE_TRANSACTION_COST = 0.0005
REALISTIC_SLIPPAGE_COST = 1.0 / 10000.0

ALPHA = 0.05


In [ ]:

def load_and_prepare_data(data_path: Path):

    df = pd.read_parquet(data_path)

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    df["block_number"] = df["block_number_y"]

    sort_cols = [c for c in ["timestamp", "block_number", "transaction_index", "log_index"] if c in df.columns]
    df = df.sort_values(sort_cols).reset_index(drop=True)

    df["price_raw"] = (df["sqrtPriceX96"].astype(float) / (2 ** 96)) ** 2
    df["price_token1_per_token0"] = df["price_raw"] * (10 ** (6 - 18))
    df["price_usdc_per_weth"] = 1.0 / df["price_token1_per_token0"].replace(0, np.nan)

    df["gas_cost_eth"] = df["gas_used"].astype(float) * df["effective_gas_price"].astype(float) / 1e18
    df["hour"] = df["timestamp"].dt.floor("h")

    hourly = (
        df.groupby("hour")
        .agg(
            open=("price_usdc_per_weth", "first"),
            close=("price_usdc_per_weth", "last"),
            high=("price_usdc_per_weth", "max"),
            low=("price_usdc_per_weth", "min"),
            median_gas_eth=("gas_cost_eth", "median"),
            swap_count=("price_usdc_per_weth", "size"),
        )
        .reset_index()
        .sort_values("hour")
        .reset_index(drop=True)
    )

    hourly["ret"] = hourly["close"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

    exec_events = (
        df.groupby("hour")
        .agg(
            exec_price=("price_usdc_per_weth", "first"),
            exec_gas_cost=("gas_cost_eth", "first"),
        )
        .reset_index()
        .sort_values("hour")
    )

    hourly = hourly.merge(exec_events, on="hour", how="left")
    hourly["exec_ret"] = hourly["exec_price"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    hourly["exec_gas_cost"] = hourly["exec_gas_cost"].fillna(0.0)

    return df, hourly


df, hourly = load_and_prepare_data(DATA_PATH)

print("Rows in event-level data:", len(df))
print("Rows in hourly data:", len(hourly))
print("Time period:", hourly["hour"].min(), "to", hourly["hour"].max())
hourly.head()


Функции

In [ ]:

def clean_signal(sig):
    return pd.Series(sig).replace([np.inf, -np.inf], np.nan).fillna(0).astype(int)


def signal_ma_crossover(data, short_window, long_window):
    fast = data["close"].rolling(short_window, min_periods=short_window).mean()
    slow = data["close"].rolling(long_window, min_periods=long_window).mean()
    sig = np.where(fast > slow, 1, np.where(fast < slow, -1, 0))
    return clean_signal(sig)


def signal_mean_reversion(data, lookback, z_threshold):
    mean = data["close"].rolling(lookback, min_periods=lookback).mean()
    std = data["close"].rolling(lookback, min_periods=lookback).std()
    z = (data["close"] - mean) / std.replace(0, np.nan)
    sig = np.where(z > z_threshold, -1, np.where(z < -z_threshold, 1, 0))
    return clean_signal(sig)


def signal_rsi(data, rsi_window, lower_threshold, upper_threshold):
    r = data["ret"].copy()
    gains = r.clip(lower=0).rolling(rsi_window, min_periods=rsi_window).mean()
    losses = (-r.clip(upper=0)).rolling(rsi_window, min_periods=rsi_window).mean()
    rs = gains / losses.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi = rsi.fillna(50)
    sig = np.where(rsi < lower_threshold, 1, np.where(rsi > upper_threshold, -1, 0))
    return clean_signal(sig)


def signal_bollinger(data, window, num_std):
    mean = data["close"].rolling(window, min_periods=window).mean()
    std = data["close"].rolling(window, min_periods=window).std()
    upper = mean + num_std * std
    lower = mean - num_std * std
    sig = np.where(data["close"] < lower, 1, np.where(data["close"] > upper, -1, 0))
    return clean_signal(sig)


def signal_breakout(data, lookback):
    high_prev = data["high"].rolling(lookback, min_periods=lookback).max().shift(1)
    low_prev = data["low"].rolling(lookback, min_periods=lookback).min().shift(1)
    sig = np.where(data["close"] > high_prev, 1, np.where(data["close"] < low_prev, -1, 0))
    return clean_signal(sig)


def signal_momentum(data, lookback, threshold):
    mom = data["close"].pct_change(lookback)
    sig = np.where(mom > threshold, 1, np.where(mom < -threshold, -1, 0))
    return clean_signal(sig)


def signal_roc(data, lookback, threshold):
    roc = data["close"].pct_change(lookback)
    sig = np.where(roc > threshold, 1, np.where(roc < -threshold, -1, 0))
    return clean_signal(sig)


def signal_volatility_breakout(data, lookback, vol_mult):
    mean = data["close"].rolling(lookback, min_periods=lookback).mean()
    vol = data["close"].rolling(lookback, min_periods=lookback).std()
    sig = np.where(
        data["close"] > mean + vol_mult * vol,
        1,
        np.where(data["close"] < mean - vol_mult * vol, -1, 0),
    )
    return clean_signal(sig)


def signal_ema_crossover(data, fast_span, slow_span):
    fast = data["close"].ewm(span=fast_span, adjust=False).mean()
    slow = data["close"].ewm(span=slow_span, adjust=False).mean()
    sig = np.where(fast > slow, 1, np.where(fast < slow, -1, 0))
    return clean_signal(sig)


def signal_channel_reversion(data, lookback):
    high_ch = data["high"].rolling(lookback, min_periods=lookback).max().shift(1)
    low_ch = data["low"].rolling(lookback, min_periods=lookback).min().shift(1)
    sig = np.where(data["close"] > high_ch, -1, np.where(data["close"] < low_ch, 1, 0))
    return clean_signal(sig)


In [ ]:

def backtest_execution_model(data, signal, model):
    out = data[["hour", "ret", "exec_ret", "exec_gas_cost"]].copy()
    out["signal"] = pd.Series(signal).values
    out["position"] = out["signal"].shift(1).fillna(0.0)
    out["position_change"] = out["position"].diff().abs().fillna(0.0)
    out["trade_flag"] = (out["position_change"] > 0).astype(int)

    if model == "Simplified":
        out["gross_ret"] = out["position"] * out["ret"]
        out["cost"] = out["position_change"] * SIMPLE_TRANSACTION_COST
    elif model == "Realistic":
        out["gross_ret"] = out["position"] * out["exec_ret"]
        out["gas_cost_return"] = np.where(out["position_change"] > 0, out["exec_gas_cost"], 0.0)
        out["slippage_cost_return"] = out["position_change"] * REALISTIC_SLIPPAGE_COST
        out["cost"] = out["gas_cost_return"] + out["slippage_cost_return"]
    else:
        raise ValueError("model must be 'Simplified' or 'Realistic'")

    out["strategy_ret"] = out["gross_ret"] - out["cost"]
    out["strategy_ret"] = out["strategy_ret"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out["equity"] = (1.0 + out["strategy_ret"]).cumprod()
    return out


def calc_sharpe(returns):
    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < 2:
        return np.nan
    std = r.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan
    return ANNUALIZATION_FACTOR * r.mean() / std


def calc_total_return(returns):
    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(r) == 0:
        return np.nan
    return (1.0 + r).prod() - 1.0


def calc_max_drawdown(returns):
    r = pd.Series(returns).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(r) == 0:
        return np.nan
    equity = (1.0 + r).cumprod()
    running_max = equity.cummax()
    dd = equity / running_max - 1.0
    return dd.min()


def period_metrics(backtest_df, start_time, end_time):
    mask = (backtest_df["hour"] >= start_time) & (backtest_df["hour"] < end_time)
    part = backtest_df.loc[mask].copy()
    return {
        "sharpe": calc_sharpe(part["strategy_ret"]),
        "total_return": calc_total_return(part["strategy_ret"]),
        "max_drawdown": calc_max_drawdown(part["strategy_ret"]),
        "number_of_trades": int(part["trade_flag"].sum()),
        "n_obs": int(len(part)),
    }


Сетки

In [ ]:

def make_grid(param_values, filter_func=None):
    keys = list(param_values.keys())
    rows = []
    for values in product(*[param_values[k] for k in keys]):
        params = dict(zip(keys, values))
        if filter_func is None or filter_func(params):
            rows.append(params)
    return rows


STRATEGIES = {
    "MA crossover": {
        "signal_func": signal_ma_crossover,
        "grid": make_grid(
            {"short_window": [12, 24, 36], "long_window": [48, 72, 120]},
            lambda p: p["short_window"] < p["long_window"],
        ),
    },
    "Mean reversion": {
        "signal_func": signal_mean_reversion,
        "grid": make_grid({"lookback": [24, 48, 72], "z_threshold": [1.0, 1.5, 2.0]}),
    },
    "RSI": {
        "signal_func": signal_rsi,
        "grid": make_grid(
            {"rsi_window": [14, 24], "lower_threshold": [30, 35], "upper_threshold": [65, 70]},
            lambda p: p["lower_threshold"] < p["upper_threshold"],
        ),
    },
    "Bollinger": {
        "signal_func": signal_bollinger,
        "grid": make_grid({"window": [20, 48], "num_std": [1.5, 2.0, 2.5]}),
    },
    "Breakout": {
        "signal_func": signal_breakout,
        "grid": make_grid({"lookback": [12, 24, 48]}),
    },
    "Momentum": {
        "signal_func": signal_momentum,
        "grid": make_grid({"lookback": [12, 24, 48], "threshold": [0.01, 0.02, 0.03]}),
    },
    "ROC": {
        "signal_func": signal_roc,
        "grid": make_grid({"lookback": [6, 12, 24], "threshold": [0.005, 0.01, 0.02]}),
    },
    "Volatility breakout": {
        "signal_func": signal_volatility_breakout,
        "grid": make_grid({"lookback": [12, 24, 48], "vol_mult": [1.0, 1.5, 2.0]}),
    },
    "EMA crossover": {
        "signal_func": signal_ema_crossover,
        "grid": make_grid(
            {"fast_span": [12, 24, 36], "slow_span": [48, 72, 120]},
            lambda p: p["fast_span"] < p["slow_span"],
        ),
    },
    "Channel reversion": {
        "signal_func": signal_channel_reversion,
        "grid": make_grid({"lookback": [24, 48, 72]}),
    },
}

for strategy_name, spec in STRATEGIES.items():
    ranges = {}
    for params in spec["grid"]:
        for key, value in params.items():
            ranges.setdefault(key, []).append(float(value))
    spec["param_ranges"] = {key: (min(values), max(values)) for key, values in ranges.items()}

for name, spec in STRATEGIES.items():
    print(name, "grid size =", len(spec["grid"]), "params =", spec["param_ranges"])



## 4. Walk-forward optimization



In [ ]:

def make_walk_forward_windows(data, train_days=30, test_days=7, step_days=7):
    start = data["hour"].min()
    end = data["hour"].max()

    train_delta = pd.Timedelta(days=train_days)
    test_delta = pd.Timedelta(days=test_days)
    step_delta = pd.Timedelta(days=step_days)

    windows = []
    cur = start
    window_id = 0

    while cur + train_delta + test_delta <= end:
        train_start = cur
        train_end = cur + train_delta
        test_start = train_end
        test_end = train_end + test_delta

        windows.append(
            {
                "window_id": window_id,
                "train_start": train_start,
                "train_end": train_end,
                "test_start": test_start,
                "test_end": test_end,
            }
        )

        cur = cur + step_delta
        window_id += 1

    return windows


windows = make_walk_forward_windows(hourly, TRAIN_DAYS, TEST_DAYS, STEP_DAYS)
print(len(windows))
windows[:3]


In [ ]:

def params_to_json(params):
    return json.dumps(params, sort_keys=True)


def optimize_and_evaluate_strategy(data, strategy_name, spec, model, windows):
    rows = []
    signal_func = spec["signal_func"]
    grid = spec["grid"]

    precomputed = []
    for params in grid:
        signal = signal_func(data, **params)
        bt = backtest_execution_model(data, signal, model)
        precomputed.append((params, bt))

    for w in windows:
        best_params = None
        best_bt = None
        best_train_sharpe = -np.inf

        for params, bt in precomputed:
            train = period_metrics(bt, w["train_start"], w["train_end"])
            train_sharpe = train["sharpe"]

            if pd.isna(train_sharpe):
                continue

            if train_sharpe > best_train_sharpe:
                best_train_sharpe = train_sharpe
                best_params = params
                best_bt = bt

        if best_bt is None:
            continue

        train = period_metrics(best_bt, w["train_start"], w["train_end"])
        test = period_metrics(best_bt, w["test_start"], w["test_end"])

        rows.append(
            {
                "strategy": strategy_name,
                "model": model,
                "window_id": w["window_id"],
                "train_start": w["train_start"],
                "train_end": w["train_end"],
                "test_start": w["test_start"],
                "test_end": w["test_end"],
                "selected_params": params_to_json(best_params),
                "is_sharpe": train["sharpe"],
                "oos_sharpe": test["sharpe"],
                "is_total_return": train["total_return"],
                "oos_total_return": test["total_return"],
                "is_max_drawdown": train["max_drawdown"],
                "oos_max_drawdown": test["max_drawdown"],
                "is_trades": train["number_of_trades"],
                "oos_trades": test["number_of_trades"],
                "is_oos_gap": train["sharpe"] - test["sharpe"],
            }
        )

    return rows


wf_rows = []
for strategy_name, spec in STRATEGIES.items():
    for model in ["Simplified", "Realistic"]:
        print("Running", strategy_name, model)
        wf_rows.extend(optimize_and_evaluate_strategy(hourly, strategy_name, spec, model, windows))

wf_results = pd.DataFrame(wf_rows)
wf_results.to_csv(OUT_DIR / "walk_forward_results.csv", index=False)
print("Saved:", OUT_DIR / "walk_forward_results.csv")
wf_results.head()


h2

In [ ]:

def normalized_param_distance(params_a, params_b, param_ranges):
    if isinstance(params_a, str):
        params_a = json.loads(params_a)
    if isinstance(params_b, str):
        params_b = json.loads(params_b)

    distances = []
    for key, (low, high) in param_ranges.items():
        a = float(params_a.get(key, low))
        b = float(params_b.get(key, low))
        denom = high - low
        if denom == 0:
            distances.append(0.0 if a == b else 1.0)
        else:
            distances.append(abs(a - b) / denom)

    if len(distances) == 0:
        return np.nan
    return float(np.mean(distances))


instability_rows = []

for strategy_name, spec in STRATEGIES.items():
    param_ranges = spec["param_ranges"]
    for model in ["Simplified", "Realistic"]:
        part = wf_results[(wf_results["strategy"] == strategy_name) & (wf_results["model"] == model)]
        part = part.sort_values("window_id").reset_index(drop=True)

        for i in range(1, len(part)):
            prev_params = part.loc[i - 1, "selected_params"]
            cur_params = part.loc[i, "selected_params"]
            instability = normalized_param_distance(prev_params, cur_params, param_ranges)

            instability_rows.append(
                {
                    "strategy": strategy_name,
                    "model": model,
                    "transition_id": int(part.loc[i, "window_id"]),
                    "prev_window_id": int(part.loc[i - 1, "window_id"]),
                    "current_window_id": int(part.loc[i, "window_id"]),
                    "parameter_instability": instability,
                    "prev_params": prev_params,
                    "current_params": cur_params,
                }
            )

instability_results = pd.DataFrame(instability_rows)
instability_results.to_csv(OUT_DIR / "parameter_instability_results.csv", index=False)
print(OUT_DIR / "parameter_instability_results.csv")
instability_results.head()



- **H1** D_perf = OOS Sharpe Simplified - OOS Sharpe Realistic
- **H2** D_stability = Parameter instability Simplified - Parameter instability Realistic
- **H3** D_gap = IS-OOS Sharpe gap Simplified - IS-OOS Sharpe gap Realistic


In [ ]:

def paired_differences_from_wf(metric_col):
    piv = wf_results.pivot_table(
        index=["strategy", "window_id"],
        columns="model",
        values=metric_col,
        aggfunc="first",
    ).reset_index()

    piv = piv.dropna(subset=["Simplified", "Realistic"])
    piv["difference"] = piv["Simplified"] - piv["Realistic"]
    return piv


def paired_differences_from_instability():
    piv = instability_results.pivot_table(
        index=["strategy", "transition_id"],
        columns="model",
        values="parameter_instability",
        aggfunc="first",
    ).reset_index()

    piv = piv.dropna(subset=["Simplified", "Realistic"])
    piv["difference"] = piv["Simplified"] - piv["Realistic"]
    return piv


def one_sided_t_test(diff_values, hypothesis, metric_name):
    d = pd.Series(diff_values).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    n = int(len(d))

    if n < 2:
        return {
            "Hypothesis": hypothesis,
            "Metric": metric_name,
            "N paired observations": n,
            "Mean difference": np.nan,
            "Median difference": np.nan,
            "Std difference": np.nan,
            "t-statistic": np.nan,
            "p-value": np.nan,
            "95% CI low": np.nan,
            "95% CI high": np.nan,
            "Decision at 5%": "Not enough observations",
        }

    mean_diff = d.mean()
    median_diff = d.median()
    std_diff = d.std(ddof=1)

    if std_diff == 0 or pd.isna(std_diff):
        t_stat = np.inf if mean_diff > 0 else -np.inf if mean_diff < 0 else 0.0
        p_value = 0.0 if mean_diff > 0 else 1.0
        ci_low = mean_diff
        ci_high = mean_diff
    else:
        se = std_diff / np.sqrt(n)
        t_stat = mean_diff / se
        p_value = stats.t.sf(t_stat, df=n - 1)
        t_crit = stats.t.ppf(0.975, df=n - 1)
        ci_low = mean_diff - t_crit * se
        ci_high = mean_diff + t_crit * se

    if p_value < 0.05 and mean_diff > 0:
        decision = "Supported at 5%"
    elif p_value < 0.10 and mean_diff > 0:
        decision = "Weak support at 10%"
    else:
        decision = "Not supported at 5%"

    return {
        "Hypothesis": hypothesis,
        "Metric": metric_name,
        "N paired observations": n,
        "Mean difference": mean_diff,
        "Median difference": median_diff,
        "Std difference": std_diff,
        "t-statistic": t_stat,
        "p-value": p_value,
        "95% CI low": ci_low,
        "95% CI high": ci_high,
        "Decision at 5%": decision,
    }


def wilcoxon_test(diff_values, hypothesis, metric_name):
    d = pd.Series(diff_values).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    d = d[d != 0]
    n = int(len(d))

    if n < 1:
        stat = np.nan
        p_value = np.nan
        decision = "Not enough non-zero differences"
    else:
        try:
            res = stats.wilcoxon(d, alternative="greater", zero_method="wilcox")
            stat = res.statistic
            p_value = res.pvalue
            if p_value < 0.05 and d.mean() > 0:
                decision = "Supported at 5%"
            elif p_value < 0.10 and d.mean() > 0:
                decision = "Weak support at 10%"
            else:
                decision = "Not supported at 5%"
        except ValueError:
            stat = np.nan
            p_value = np.nan
            decision = "Test failed"

    return {
        "Hypothesis": hypothesis,
        "Metric": metric_name,
        "N non-zero differences": n,
        "Wilcoxon statistic": stat,
        "Wilcoxon p-value": p_value,
        "Wilcoxon decision": decision,
    }


h1_diff = paired_differences_from_wf("oos_sharpe")
h2_diff = paired_differences_from_instability()
h3_diff = paired_differences_from_wf("is_oos_gap")

h1_diff.to_csv(OUT_DIR / "h1_oos_sharpe_differences.csv", index=False)
h2_diff.to_csv(OUT_DIR / "h2_parameter_instability_differences.csv", index=False)
h3_diff.to_csv(OUT_DIR / "h3_is_oos_gap_differences.csv", index=False)

summary_rows = [
    one_sided_t_test(
        h1_diff["difference"],
        "H1",
        "OOS Sharpe: Simplified - Realistic",
    ),
    one_sided_t_test(
        h2_diff["difference"],
        "H2",
        "Parameter instability: Simplified - Realistic",
    ),
    one_sided_t_test(
        h3_diff["difference"],
        "H3",
        "IS-OOS Sharpe gap: Simplified - Realistic",
    ),
]

wilcoxon_rows = [
    wilcoxon_test(
        h1_diff["difference"],
        "H1",
        "OOS Sharpe: Simplified - Realistic",
    ),
    wilcoxon_test(
        h2_diff["difference"],
        "H2",
        "Parameter instability: Simplified - Realistic",
    ),
    wilcoxon_test(
        h3_diff["difference"],
        "H3",
        "IS-OOS Sharpe gap: Simplified - Realistic",
    ),
]

hypothesis_summary = pd.DataFrame(summary_rows)
hypothesis_wilcoxon = pd.DataFrame(wilcoxon_rows)

hypothesis_summary.to_csv(OUT_DIR / "hypothesis_test_summary_ttest.csv", index=False)
hypothesis_wilcoxon.to_csv(OUT_DIR / "hypothesis_test_summary_wilcoxon.csv", index=False)


hypothesis_summary.round(4)


In [ ]:

hypothesis_wilcoxon.round(4)
